# Sevastopol AI — запуск главного бота в Colab

Пять шагов, всё в этом блокноте. Нужны две вещи: **токен бота** от @BotFather и
**свой Telegram ID** (узнать: написать @userinfobot).

Бот читает таблицу из Google Sheets. Если таблица закрыта для чтения по ссылке,
он возьмёт лежащую в репозитории копию `Sevastopol AI База.xlsx` — стартует в
любом случае, но данные будут из файла, а не из твоей таблицы.

Тест-бот на виджетах живёт отдельно, в папке `testwidgetsbot/` — этот блокнот
его не касается.

## Шаг 1. Код и библиотеки

Скачивает репозиторий (или обновляет уже скачанный) и ставит библиотеки.
Повторный запуск безопасен: обновит и оставит всё как есть.

In [ ]:
%cd /content
!test -d SevastopolAIbot && (git -C SevastopolAIbot fetch -q origin main && git -C SevastopolAIbot reset -q --hard origin/main) || git clone -q https://github.com/Yurich-citycode/SevastopolAIbot.git
%cd /content/SevastopolAIbot
!pip install -q -r requirements.txt
!git log --oneline -1
print("готово")

## Шаг 2. Токен и админ

Токен в код не вписывай. Если сохранил в Colab (значок ключа слева → Secrets →
имя `TG_TOKEN`), ячейка возьмёт его сама; если нет — спросит полем ввода.

`ADMIN_ID` — твой Telegram ID, нужен для `/admin` и заявок с формы. Можно
оставить пустым: бот запустится, просто эти функции будут выключены.

In [ ]:
import os, re
from getpass import getpass

def secret(name):
    try:
        from google.colab import userdata
        return str(userdata.get(name) or "").strip()
    except Exception:
        return ""

token = secret("TG_TOKEN")
if not token:
    token = getpass("Вставь токен бота и нажми Enter: ").strip()
os.environ["TG_TOKEN"] = token

admin = secret("ADMIN_ID")
if not admin:
    admin = getpass("Твой Telegram ID (Enter — пропустить): ").strip()
os.environ["ADMIN_ID"] = admin

looks_right = bool(re.fullmatch(r"\d{6,12}:[A-Za-z0-9_-]{30,}", token))
print("токен:", "есть" if token else "пусто", "· длина", len(token),
      "· формат", "похож" if looks_right else "НЕ похож")
print("админ:", admin or "не задан — /admin и заявки будут выключены")
if token and not looks_right:
    print("Токен выглядит как 8123456789:AAH... — проверь, что скопировал его целиком.")

## Шаг 3. Таблица: что увидит бот

Проверяет ту же копию, которую читает бот: скачивает её из Google Sheets и
смотрит по разделам — сколько строк, есть ли пустые названия, битые координаты,
прошедшие даты, слишком длинные описания.

Ошибка → раздел в боте будет пустым или кривым. Предупреждение → на твоё
усмотрение. Строки без названия бот просто пропускает, это нормально.

In [ ]:
!cd /content/SevastopolAIbot && python tools/check_base.py --sheets

Если Google не отдал таблицу — проверь доступ: в таблице **Файл → Настройки
доступа → Все, у кого есть ссылка → Читатель**. Проверить локальную копию из
репозитория можно без Google: `python tools/check_base.py`.

## Шаг 4. Запуск

Ячейка не завершится — бот работает, пока она крутится. Так и должно быть.

В логах сразу видно, откуда данные:

```
🔄 Кэш: обновление из Google Sheets
Кэш 'Где поесть': 145 строк
Кэш 'Локации': 53 строк
```

Если вместо этого `Google Sheets недоступен — беру локальную копию`, значит
таблица закрыта, и бот работает на файле из репозитория.

In [ ]:
import subprocess, sys

DIR = "/content/SevastopolAIbot"
p = subprocess.Popen([sys.executable, "-u", "sevastopolaibot.py"], cwd=DIR, env=dict(os.environ),
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
print("\n=== бот завершился с кодом", p.wait(), "===")

## Шаг 5. Проверить в Telegram

1. Открой своего бота по юзернейму → **Start**.
2. Должны прийти: согласие на обработку данных, панель управления, приветствие
   и меню: 🍽 Где поесть · 📍 Локации · 📅 События · 🗺 Маршруты · 🗺 Живая карта
   · 📰 Новости · ✍️ Предложить место.
3. Походи по разделам: «Где поесть» → «Кофе» → «Все списком» → карточка.

Что должно работать сразу: разделы, карточки мест, паспорт и XP, заявка с формы.
«Живая карта» скажет, что сервис не найден, — карта поднимается отдельно, на
своём компьютере (`RUN.bat`) или на сервере.

| Симптом | Что делать |
|---|---|
| `Unauthorized` | токен скопирован не целиком: @BotFather → `/mybots` → API Token |
| пустые разделы | таблица не прочиталась: смотри Шаг 3 и строки `Кэш ...` в логах |
| «На ближайшие дни событий не найдено» | во вкладке «События» нет будущих дат: добавь даты |
| `Не удалось загрузить таблицу при старте` | Google не отдал копию, бот работает на локальном файле |
| ячейка «спит», бот молчит | бесплатный Colab засыпает: перезапусти ячейку запуска |

Проверить бота без Telegram, прямо в консоли — `python tools/try_bot.py`:
пишешь `/start`, `!cat_coffee`, `/route Фиолент`, он отвечает и печатает кнопки.